In [2]:
import pandas as pd
import re

# Load file CSV
df = pd.read_csv('master_chat_wa.csv')

# Menampilkan 5 data teratas dan informasi kolom
print(df.info())
df[['Message Body']].head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3175 entries, 0 to 3174
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Country         3175 non-null   object
 1   Phone Number    3175 non-null   int64 
 2   Formatted Name  3175 non-null   object
 3   Display Name    3091 non-null   object
 4   Message Time    3175 non-null   object
 5   Message Type    3175 non-null   object
 6   Message Body    3120 non-null   object
 7   Message Id      3175 non-null   object
dtypes: int64(1), object(7)
memory usage: 198.6+ KB
None


,Message Body
0,【IMAGE】
1,Anugerah komputer
2,Oke bg
3,Bg berapa kemarin harganya ya
4,Nnti tnya y bg blom dtg bos nya


In [3]:
def auto_label_chat(text):
    if pd.isna(text):
        return -1
    
    text = str(text).lower()
    
    # Kata kunci penanda persetujuan
    keywords_setuju = [
        'oke', 'ok', 'siap', 'acc', 'lanjut', 'proses', 'bisa', 'boleh', 
        'transfer', 'deal', 'ambil', 'kirim', 'jadi', 'minta', 'mau'
    ]
    
    # Kata kunci penanda pembatalan
    keywords_batal = [
        'batal', 'cancel', 'gajadi', 'ga jadi', 'ndak jadi', 'tidak', 
        'tunda', 'ganti', 'maaf', 'belum', 'pending', 'salah'
    ]
    
    # Cek kecocokan kata kunci
    for word in keywords_batal:
        if word in text:
            return 0  # Label Batal
            
    for word in keywords_setuju:
        if word in text:
            return 1  # Label Setuju
            
    return -1  # Tetap tandai sebagai Lainnya / Perlu Review Manual

# Terapkan fungsi ke data yang labelnya masih "Lainnya"
df['Target_Label'] = df['Message Body'].apply(auto_label_chat)

# Filter data yang berhasil dilabeli (0 dan 1) untuk masuk ke tahap pelatihan NLP
data_training = df[df['Target_Label'].isin([0, 1])].copy()

print(f"Total data siap latih setelah auto-labeling: {len(data_training)} baris")
print(data_training['Target_Label'].value_counts())

Total data siap latih setelah auto-labeling: 1442 baris
Target_Label
1    1215
0     227
Name: count, dtype: int64


In [4]:
def auto_label_chat(text):
    if pd.isna(text):
        return -1
    
    text = str(text).lower()
    
    # Kata kunci penanda persetujuan
    keywords_setuju = [
        'oke', 'ok', 'siap', 'acc', 'lanjut', 'proses', 'bisa', 'boleh', 
        'transfer', 'deal', 'ambil', 'kirim', 'jadi', 'minta', 'mau'
    ]
    
    # Kata kunci penanda pembatalan
    keywords_batal = [
        'batal', 'cancel', 'gajadi', 'ga jadi', 'ndak jadi', 'tidak', 
        'tunda', 'ganti', 'maaf', 'belum', 'pending', 'salah'
    ]
    
    # Cek kecocokan kata kunci
    for word in keywords_batal:
        if word in text:
            return 0  # Label Batal
            
    for word in keywords_setuju:
        if word in text:
            return 1  # Label Setuju
            
    return -1  # Tetap tandai sebagai Lainnya / Perlu Review Manual

# Terapkan fungsi ke data yang labelnya masih "Lainnya"
df['Target_Label'] = df['Message Body'].apply(auto_label_chat)

# Filter data yang berhasil dilabeli (0 dan 1) untuk masuk ke tahap pelatihan NLP
data_training = df[df['Target_Label'].isin([0, 1])].copy()

print(f"Total data siap latih setelah auto-labeling: {len(data_training)} baris")
print(data_training['Target_Label'].value_counts())

Total data siap latih setelah auto-labeling: 1442 baris
Target_Label
1    1215
0     227
Name: count, dtype: int64


In [5]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    # Hapus penanda sistem seperti 【IMAGE】 atau emoji bawaan log
    text = re.sub(r'【.*?】', '', text)
    # Hapus tanda baca dan angka, sisakan huruf saja
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Hapus spasi berlebih
    text = re.sub(r'\s+', ' ', text).strip()
    return text

data_training['Cleaned_Message'] = data_training['Message Body'].apply(clean_text)
data_training[['Message Body', 'Cleaned_Message', 'Target_Label']].head(10)

,Message Body,Cleaned_Message,Target_Label
2,Oke bg,oke bg,1
5,Okey bg,okey bg,1
9,Okey bg bntr lagi saya kesana,okey bg bntr lagi saya kesana,1
11,Boleh ikutin tutorial ini pak,boleh ikutin tutorial ini pak,1
12,misa bg sya kemarin servis leptop ditempat abg...,misa bg sya kemarin servis leptop ditempat abg...,1
15,Buka kak tapi sampai sore. Soalnya toko mau tu...,buka kak tapi sampai sore soalnya toko mau tut...,1
16,oke nnti saya bawa,oke nnti saya bawa,1
17,Oke kak,oke kak,1
18,bg mau tanya leptop nya udh betul belum,bg mau tanya leptop nya udh betul belum,0
19,Blm kak. Kami Senin tadi baru buka. Listrik ka...,blm kak kami senin tadi baru buka listrik kami...,0


In [9]:
import fasttext
import numpy as np

# 1. Simpan teks bersih ke file txt sementara untuk dibaca FastText
with open('corpus_chat.txt', 'w', encoding='utf-8') as f:
    for text in data_training['Cleaned_Message']:
        # Hanya ambil teks yang tidak kosong
        if str(text).strip():
            f.write(str(text) + '\n')

# 2. Latih model FastText (unsupervised) untuk mengenali pola kata tokomu
# dim=100 berarti setiap kalimat akan diubah menjadi 100 deret angka
print("Sedang melatih model representasi bahasa FastText...")
ft_model = fasttext.train_unsupervised('corpus_chat.txt', model='cbow', dim=100, epoch=20)
print("Selesai!")

# 3. Fungsi untuk mengubah setiap kalimat di dataframe menjadi vektor angka
def sentence_to_vector(text):
    return ft_model.get_sentence_vector(str(text))

# 4. Terapkan fungsi vektorisasi ke seluruh baris data
data_training['Vector'] = data_training['Cleaned_Message'].apply(sentence_to_vector)

# Siapkan X (Fitur Vektor) dan y (Label Target) untuk SVM
X = np.vstack(data_training['Vector'].values)
y = data_training['Target_Label'].astype(int).values

print(f"Bentuk data X (Vektor): {X.shape}")
print(f"Bentuk data y (Label): {y.shape}")

Sedang melatih model representasi bahasa FastText...
Selesai!
Bentuk data X (Vektor): (1442, 100)
Bentuk data y (Label): (1442,)


In [10]:
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, accuracy_score
import pickle

# 1. Bagi data: 80% untuk Latih, 20% untuk Uji/Testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Inisialisasi Model SVM
# probability=True SANGAT PENTING agar kita bisa menarik nilai persentase untuk dilempar ke LLM nanti
print("Sedang melatih model SVM...")
svm_model = SVC(kernel='linear', probability=True, random_state=42)
svm_model.fit(X_train, y_train)

# 3. Evaluasi Prediksi pada 20% Data Uji
y_pred = svm_model.predict(X_test)
akurasi = accuracy_score(y_test, y_pred)

print(f"\n--- HASIL EVALUASI MODEL ---")
print(f"Akurasi Keseluruhan: {akurasi * 100:.2f}%")
print("\nDetail Laporan Klasifikasi:")
print(classification_report(y_test, y_pred, target_names=['Batal (0)', 'Setuju (1)']))

# 4. Simpan model SVM dan FastText agar nanti bisa dipakai di API (FastAPI)
ft_model.save_model("fasttext_model.bin")
with open("svm_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)
print("\nModel FastText dan SVM berhasil disimpan!")

Sedang melatih model SVM...

--- HASIL EVALUASI MODEL ---
Akurasi Keseluruhan: 84.08%

Detail Laporan Klasifikasi:
              precision    recall  f1-score   support

   Batal (0)       0.00      0.00      0.00        46
  Setuju (1)       0.84      1.00      0.91       243

    accuracy                           0.84       289
   macro avg       0.42      0.50      0.46       289
weighted avg       0.71      0.84      0.77       289



c:\Users\peno\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\peno\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\peno\anaconda3\envs\nlp_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.sh


Model FastText dan SVM berhasil disimpan!
